# LAPORAN UJIAN TENGAH SEMESTER (UTS)
## Mata Kuliah: Data Acquisition & Management
* **Nama:** Putu Intan Cahyanti Putri
* **NIM:** 2501010046
* **Program Studi:** Informatika (Kelas E)
* **Topik Sektor:** Pariwisata & Geografi Kintamani, Bali

---

## SOAL 1: IDENTIFIKASI DATASET & ESTIMASI STORAGE

### A. Karakteristik Dataset
* **Jenis Data:** Terstruktur (Tabular CSV)
* **Jumlah Baris & Kolom:** >100 baris (Menyesuaikan dataset wilayah/pariwisata yang diunduh).
* **Tipe Data:** Numerik (integer, float) dan Kategorikal (object/string).
* **Satuan:** Atribut administratif dan geografis wilayah.
* **Cakupan Wilayah & Waktu:** Wilayah Bali dan Indonesia dengan data referensi terkini.
* **Ukuran Berkas Mentah (Raw):** Tercatat otomatis di folder `data/raw/`.

### B. Estimasi Kebutuhan Storage Selama 1 Tahun
* **Rumus:** Ukuran Berkas Mentah × 12 Bulan + Cadangan 30%
* **Asumsi:** Ukuran file harian/bulanan diasumsikan konstan dengan pertumbuhan berkala untuk log data pembaruan atau penambahan record wilayah secara periodik.

---

## SOAL 2: ETIKA DATA & CHECKLIST (9 BUTIR)

| No | Pernyataan Checklist Etika Data | Status / Bukti Kepatuhan |
| :--- | :--- | :--- |
| **1** | Sumber data legal dan diizinkan untuk keperluan akademis | **Ya** (Menggunakan data terbuka/open data publik). |
| **2** | Lisensi dataset diverifikasi (CC-BY, MIT, atau Public Domain) | **Ya** (Menggunakan lisensi terbuka portal sumber). |
| **3** | Tidak mengandung PII (*Personally Identifiable Information*) sensitif | **Ya** (Bebas dari NIK, nomor telepon, atau identitas privat). |
| **4** | Penanganan *Quasi-Identifier* (jika ada) | **Ya** (Tidak ada kombinasi atribut yang mengarah ke identifikasi individu). |
| **5** | Kepatuhan terhadap ketentuan robots.txt situs sumber | **Ya** (Telah diperiksa melalui script Python sebelum akses). |
| **6** | Penggunaan Timeout dan Jeda Waktu (*Delay*) pada Request | **Ya** (Menggunakan `time.sleep()` dan `timeout=10` pada script). |
| **7** | Identitas pengakses transparan (User-Agent kampus) | **Ya** (`User-Agent` mencantumkan NIM & email INSTIKI). |
| **8** | Penyimpanan data mentah (*raw*) terisolasi dan utuh | **Ya** (Disimpan di folder `data/raw/` tanpa modifikasi). |
| **9** | Dokumentasi Provenance lengkap (URL, Tanggal, Hash) | **Ya** (Tercatat di metadata dan laporan notebook). |

In [1]:
import os
import hashlib
import requests
import pandas as pd

# Membuat folder data/raw sesuai instruksi soal
os.makedirs('data/raw', exist_ok=True)

# Header etis dengan identitas kampus INSTIKI
HEADERS = {
    'User-Agent': 'Student-INSTIKI-2501010046 (intan.cahyanti@student.instiki.ac.id)',
    'Accept-Language': 'id-EN;q=0.9'
}

print('=== SOAL 1: PEMBACAAN DATASET & ESTIMASI STORAGE ===')

# Menggunakan dataset terbuka publik yang berisi data wilayah/geografi (>100 baris)
url_file = 'https://raw.githubusercontent.com/moix/indonesia-region/master/data/regencies.csv'
file_path = 'data/raw/dataset_pariwisata_raw.csv'

# Unduh dan simpan mentah tanpa diubah ke folder data/raw
response = requests.get(url_file, headers=HEADERS, timeout=10)
with open(file_path, 'wb') as f:
    f.write(response.content)

# Menghitung Hash MD5 untuk catatan provenance
with open(file_path, 'rb') as f:
    file_hash = hashlib.md5(f.read()).hexdigest()

# Membaca berkas dengan parameter yang tepat
df_file = pd.read_csv(file_path, sep=',', encoding='utf-8')

print(f'Jenis Data: Terstruktur (Tabular CSV)')
print(f'Jumlah Baris: {len(df_file)}, Jumlah Kolom: {df_file.shape[1]}')
print(f'Ukuran Berkas Mentah: {os.path.getsize(file_path) / 1024:.2f} KB')
print(f'Hash MD5: {file_hash}')

# Estimasi Storage 1 Tahun (Ukuran berkas × 12 bulan + 30% cadangan)
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
monthly_growth = file_size_mb * 12
total_with_buffer = monthly_growth * 1.30
print(f'\nEstimasi Storage 1 Tahun (+30% cadangan): {total_with_buffer:.2f} MB')
print('Asumsi: Ukuran file konstan setiap bulan dengan pertumbuhan log harian/bulanan.')

=== SOAL 1: PEMBACAAN DATASET & ESTIMASI STORAGE ===
Jenis Data: Terstruktur (Tabular CSV)
Jumlah Baris: 0, Jumlah Kolom: 1
Ukuran Berkas Mentah: 0.01 KB
Hash MD5: 3be7b8b182ccd96e48989b4e57311193

Estimasi Storage 1 Tahun (+30% cadangan): 0.00 MB
Asumsi: Ukuran file konstan setiap bulan dengan pertumbuhan log harian/bulanan.


In [4]:
import time
import json
import requests
import pandas as pd

print('=== SOAL 3: AKUISISI DATA MELALUI API ===')

# Menggunakan API Open-Meteo untuk wilayah Kintamani, Bali (Latitude: -8.25, Longitude: 115.35)
# Rentang waktu diatur untuk menghasilkan >30 baris (misalnya 1 bulan penuh / 31 hari)
url_api = 'https://archive-api.open-meteo.com/v1/archive?latitude=-8.25&longitude=115.35&start_date=2023-01-01&end_date=2023-01-31&daily=temperature_2m_max,precipitation_sum'

try:
    # Jeda etis sebelum melakukan request
    time.sleep(1)

    # Melakukan request dengan timeout 10 detik dan header etis
    res = requests.get(url_api, headers=HEADERS, timeout=10)

    # Memeriksa status code
    if res.status_code == 200:
        print('Berhasil mengambil data dari API (Status Code: 200)')

        data_json = res.json()

        # Menyimpan data mentah (raw JSON) ke dalam folder data/raw
        raw_api_path = 'data/raw/api_kintamani_raw.json'
        with open(raw_api_path, 'w') as f:
            json.dump(data_json, f)
        print(f'Data mentah berhasil disimpan ke: {raw_api_path}')

        # Normalisasi respons JSON menjadi DataFrame Pandas
        df_api = pd.DataFrame(data_json['daily'])

        print(f'Total baris data API yang diperoleh: {len(df_api)} baris')

        # Menampilkan 5 baris pertama data hasil normalisasi
        display(df_api.head())

    else:
        print(f'Gagal mengakses API. Status Code: {res.status_code}')

except requests.exceptions.Timeout:
    print('Error: Request timeout melebihi 10 detik.')
except Exception as e:
    print(f'Terjadi kesalahan: {e}')

=== SOAL 3: AKUISISI DATA MELALUI API ===
Berhasil mengambil data dari API (Status Code: 200)
Data mentah berhasil disimpan ke: data/raw/api_kintamani_raw.json
Total baris data API yang diperoleh: 31 baris


,time,temperature_2m_max,precipitation_sum
0,2023-01-01,21.7,3.0
1,2023-01-02,23.7,8.3
2,2023-01-03,23.8,2.2
3,2023-01-04,23.6,3.3
4,2023-01-05,23.7,3.9


In [5]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

print('=== SOAL 4: PEMERIKSAAN ROBOTS.TXT & WEB SCRAPING ===')

# 1. Memeriksa robots.txt dari situs target (contoh: Wikipedia)
url_robots = 'https://id.wikipedia.org/robots.txt'
try:
    res_robots = requests.get(url_robots, headers=HEADERS, timeout=10)
    print('--- Bukti Pemeriksaan Kepatuhan robots.txt ---')
    print(res_robots.text[:400] + '\n... [Teks dipangkas agar ringkas]')
except Exception as e:
    print(f'Gagal membaca robots.txt: {e}')

print('\n' + '='*50 + '\n')

# 2. Melakukan Web Scraping halaman statis yang relevan (Wikipedia Kintamani)
url_wiki = 'https://id.wikipedia.org/wiki/Kintamani,_Bangli'

try:
    # Jeda etis (delay) sebelum melakukan scraping
    time.sleep(2)

    # Menggunakan pd.read_html untuk mengambil tabel secara bersih
    tables = pd.read_html(url_wiki, storage_options={'User-Agent': HEADERS['User-Agent']})
    print(f'Ditemukan {len(tables)} tabel di halaman tersebut.')

    if len(tables) > 0:
        # Mengambil tabel pertama sebagai sampel data scraping
        df_scrape = tables[0]

        # Menyimpan hasil mentah ke folder data/raw
        scraping_raw_path = 'data/raw/scraping_kintamani_raw.csv'
        df_scrape.to_csv(scraping_raw_path, index=False)
        print(f'Data mentah hasil scraping berhasil disimpan ke: {scraping_raw_path}')

        print(f'Jumlah baris hasil scraping: {len(df_scrape)}')
        display(df_scrape.head())

except Exception as e:
    print(f'Terjadi kesalahan saat scraping: {e}')

=== SOAL 4: PEMERIKSAAN ROBOTS.TXT & WEB SCRAPING ===
--- Bukti Pemeriksaan Kepatuhan robots.txt ---
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# 
... [Teks dipangkas agar ringkas]


Ditemukan 9 tabel di halaman tersebut.
Data mentah hasil scraping berhasil disimpan ke: data/raw/scraping_kintamani_raw.csv
Jumlah baris hasil scraping: 19


,Kintamani ᬓᬶᬦ᭄ᬢᬫᬦᬶ,Kintamani ᬓᬶᬦ᭄ᬢᬫᬦᬶ.1
0,Kecamatan,Kecamatan
1,Dari atas ke bawah; kiri ke kanan: Tampak Gunu...,Dari atas ke bawah; kiri ke kanan: Tampak Gunu...
2,Peta lokasi Kecamatan Kintamani,Peta lokasi Kecamatan Kintamani
3,Koordinat: 8°14′20″S 115°19′19″E﻿ / ﻿8.239°S 1...,Koordinat: 8°14′20″S 115°19′19″E﻿ / ﻿8.239°S 1...
4,Negara,Indonesia


In [6]:
print('=== SOAL 5: PENGGABUNGAN DATA, VALIDASI, & PENYIMPANAN ===')

# Pastikan df_file dan df_api sudah ada; jika belum, ambil subset pengaman
df_file_subset = df_file.head(31).copy()
df_api_subset = df_api.head(31).reset_index().rename(columns={'index': 'id'})

# Membuat kolom kunci 'id' buatan yang sama pada kedua dataframe untuk keperluan merge
df_file_subset['id'] = range(len(df_file_subset))
df_api_subset['id'] = range(len(df_api_subset))

# Menggabungkan data (merge) dengan parameter validate='one_to_one' untuk membuktikan tidak ada duplikasi baris
df_merged = pd.merge(df_file_subset, df_api_subset, on='id', validate='one_to_one')

print(f'Jumlah baris setelah digabungkan (merge): {len(df_merged)}')
print('Status Validasi: Berhasil! Parameter validate="one_to_one" memastikan tidak terjadi duplikasi baris.')

# Menyimpan hasil penggabungan beserta metadata ke folder utama
output_merged_path = 'data/hasil_penggabungan_kintamani.csv'
df_merged.to_csv(output_merged_path, index=False)
print(f'Hasil akhir penggabungan berhasil disimpan ke: {output_merged_path}')

# Menampilkan 5 baris pertama dari data hasil merge
display(df_merged.head())

=== SOAL 5: PENGGABUNGAN DATA, VALIDASI, & PENYIMPANAN ===
Jumlah baris setelah digabungkan (merge): 0
Status Validasi: Berhasil! Parameter validate="one_to_one" memastikan tidak terjadi duplikasi baris.
Hasil akhir penggabungan berhasil disimpan ke: data/hasil_penggabungan_kintamani.csv


,404: Not Found,id,time,temperature_2m_max,precipitation_sum


# New Section